## Пример на pyspark

В качестве набора данных для примера будем использовать данные конкурса про ответы студентов на тесты
https://www.kaggle.com/c/riiid-test-answer-prediction

При подключении к spark драйверу установим лимиты по памяти и по числу ядер. Также выберем номер порта для Spark UI

Нужно выбрать уникальное имя приложения и номер порта, чтобы не войти в коллизию с другими пользователями

In [1]:
%pylab inline
import pandas as pd
import numpy as np

import os
import sys

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
# Указываем переменные окружения
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Добавим Spark Python API в sys.path
sys.path.append("/opt/homebrew/opt/apache-spark/libexec/python")
sys.path.append("/opt/homebrew/opt/apache-spark/libexec/python/lib/py4j-0.10.9.7-src.zip") 

In [3]:
!java -version

openjdk version "17.0.15" 2025-04-15
OpenJDK Runtime Environment Homebrew (build 17.0.15+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.15+0, mixed mode, sharing)


In [4]:
!spark-submit --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.0
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.15
Branch HEAD
Compiled by user wenchen on 2025-05-19T07:58:03Z
Revision fa33ea000a0bda9e5a3fa1af98e8e85b8cc5e4d4
Url https://github.com/apache/spark
Type --help for more information.


In [5]:
# !pip install findspark

In [6]:
import findspark
findspark.init()

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .config("spark.dynamicAllocation.enabled", "true")
        .config("spark.executor.memory", "2g")
        .config("spark.driver.memory", "1g")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/02 16:13:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Данные будем читать из заранее сконвертированного parquet

In [8]:
riiid_df = spark.read.parquet("riiid/lectures.parquet",)

Схема данных и первые 10 записей

In [10]:
riiid_df.printSchema()

root
 |-- lecture_id: integer (nullable = true)
 |-- tag: integer (nullable = true)
 |-- part: integer (nullable = true)
 |-- type_of: string (nullable = true)



In [11]:
riiid_df.show(10)

+----------+---+----+----------------+
|lecture_id|tag|part|         type_of|
+----------+---+----+----------------+
|     24402|149|   1|         concept|
|     30455| 44|   6|         concept|
|      2918| 78|   5|         concept|
|     11236| 61|   1|solving question|
|     26335|170|   5|         concept|
|      8708| 47|   5|         concept|
|     10180| 33|   6|         concept|
|     11772|113|   4|solving question|
|     20590|145|   7|         concept|
|      5990| 67|   4|         concept|
+----------+---+----+----------------+
only showing top 10 rows


In [16]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

In [ ]:
# Определение схемы
schema = StructType([
    StructField("lecture_id", IntegerType(), True),
    StructField("tag", IntegerType(), True),
    StructField("part", IntegerType(), True),
    StructField("type_of", StringType(), True)
])

In [22]:
# Чтение Parquet с явной схемой
df = spark.read.schema(schema).parquet("riiid/lectures.parquet")

In [23]:
df.printSchema()

root
 |-- lecture_id: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- part: integer (nullable = true)
 |-- type_of: string (nullable = true)



In [24]:
df.show(10)

25/09/02 16:22:57 ERROR Executor: Exception in task 0.0 in stage 11.0 (TID 11)
org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///Users/stureiko/Documents/Programming/Otus/MLOps/18%20-%20Подготовка%20и%20извлечение%20данных/code/sources/riiid/lectures.parquet/part-00000-387d6bfe-62c1-4fe8-9cde-b89ccc4cc9a2-c000.snappy.parquet. Data type mismatches when reading Parquet column [tag]. Expected Spark type string, actual Parquet type INT32. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.parquetColumnDataTypeMismatchError(QueryExecutionErrors.scala:847)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:138)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:695)
	at org.apache.spark.sql.catalyst.

Py4JJavaError: An error occurred while calling o130.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///Users/stureiko/Documents/Programming/Otus/MLOps/18%20-%20Подготовка%20и%20извлечение%20данных/code/sources/riiid/lectures.parquet/part-00000-387d6bfe-62c1-4fe8-9cde-b89ccc4cc9a2-c000.snappy.parquet. Data type mismatches when reading Parquet column [tag]. Expected Spark type string, actual Parquet type INT32. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.parquetColumnDataTypeMismatchError(QueryExecutionErrors.scala:847)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:138)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:695)
	at org.apache.spark.sql.catalyst.expressions.G

In [26]:
df = spark.read.parquet("riiid/lectures.parquet")
df = df.withColumn("tag", df["tag"].cast("string"))
df.printSchema()

root
 |-- lecture_id: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- part: integer (nullable = true)
 |-- type_of: string (nullable = true)



In [27]:
df.show(10)

+----------+---+----+----------------+
|lecture_id|tag|part|         type_of|
+----------+---+----+----------------+
|     24402|149|   1|         concept|
|     30455| 44|   6|         concept|
|      2918| 78|   5|         concept|
|     11236| 61|   1|solving question|
|     26335|170|   5|         concept|
|      8708| 47|   5|         concept|
|     10180| 33|   6|         concept|
|     11772|113|   4|solving question|
|     20590|145|   7|         concept|
|      5990| 67|   4|         concept|
+----------+---+----+----------------+
only showing top 10 rows


In [12]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter
riiid_df

lecture_id,tag,part,type_of
24402,149,1,concept
30455,44,6,concept
2918,78,5,concept
11236,61,1,solving question
26335,170,5,concept
8708,47,5,concept
10180,33,6,concept
11772,113,4,solving question
20590,145,7,concept
5990,67,4,concept


Замерим время выполнения простых запросов с группировками

In [13]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col

In [14]:
%%time
(
riiid_df
    .select('content_id', 'answered_correctly')
    .groupBy('content_id')
    .mean('answered_correctly')
    .show()
)

{"ts": "2025-09-02 16:14:53.424", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `content_id` cannot be resolved. Did you mean one of the following? [`lecture_id`, `part`, `tag`, `type_of`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o37.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `content_id` cannot be resolved. Did you mean one of the following? [`lecture_id`, `part`, `tag`, `type_of`]. SQLSTATE: 42703;\n'Project ['content_id, 'answered_correctly]\n+- Relation [lecture_id#0,tag#1,part#2,type_of#3] parquet\n\n\tat org.apache.spark.sql.errors.QueryCom

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `content_id` cannot be resolved. Did you mean one of the following? [`lecture_id`, `part`, `tag`, `type_of`]. SQLSTATE: 42703;
'Project ['content_id, 'answered_correctly]
+- Relation [lecture_id#0,tag#1,part#2,type_of#3] parquet


In [10]:
%%time
(
riiid_df
    .select('user_id', 'answered_correctly')
    .where(col('answered_correctly') != -1)
    .groupby('user_id')
    .mean('answered_correctly')
    .show()
)

+---------+-----------------------+
|  user_id|avg(answered_correctly)|
+---------+-----------------------+
|252345392|       0.56480117820324|
|253500385|     0.5285296981499513|
|242039738|    0.46367041198501874|
|254408119|     0.7657534246575343|
|224426519|     0.7173333333333334|
|240485154|     0.5521978021978022|
|244175802|     0.7359550561797753|
|211646563|     0.7669491525423728|
|219176829|     0.5384615384615384|
|218611655|      0.676829268292683|
|212559006|                  0.525|
|259875659|     0.6183574879227053|
|240750690|     0.6890756302521008|
|245081718|    0.30612244897959184|
|254957588|     0.2702702702702703|
|226534187|     0.6363636363636364|
|257509511|    0.49019607843137253|
|220663840|     0.7661870503597122|
|252861630|               0.546875|
|243302977|     0.6267942583732058|
+---------+-----------------------+
only showing top 20 rows

CPU times: user 5.73 ms, sys: 989 µs, total: 6.71 ms
Wall time: 13.1 s


## Упражнение 1
Выведите top 10 студентов с наилучшими результатами. 
Обратите внимание, что поле answered_correctly равно -1, если это была лекция, а не тест. Такие записи нужно исключить.

## Упражнение 2
Выведите top 10 задач с наихудшими результатами

## pyspark user defined functions (UDF)

Как и для других языков, поддерживаемых Spark, для python есть возможность использовать UDF. При этом возникают дополнительные накладные расходы по сравнению с Java и Scala на маршалинг данных.

In [ ]:
from pyspark.sql.types import LongType

def to_months(ms):
    return ms // 31536000000 // 12 #1 year = 31536000000 ms

to_months_udf = f.udf(to_months, LongType())

Замерим время выполнения без UDF

In [ ]:
%%time
(
    riiid_df
        .select("content_id", "timestamp")
        .groupby("content_id")
        .mean("timestamp")
        .show()
)

Применим простой UDF к похожему запросу

In [ ]:
%%time
(
    riiid_df
        .select("content_id", to_days_udf("timestamp").alias("months"))
        .groupBy("content_id")
        .mean("months")
        .show()
)

Перепишем логику, которая была в UDF

In [ ]:
%%time
(
 riiid_df
    .select("content_id", (col("timestamp") / 31536000000 / 12).alias("months"))
    .groupby("content_id")
    .mean("months")
    .show()
)

## Упражнение 4
Постройте гистограмму по числу месяцев до первого взаимодействия студента с заданием

## Обогащение данных

Таблица с вопросами лежит в отдельном файле questions.csv. 

In [ ]:
questions = spark.read.csv("riiid/questions.csv", header=True, inferSchema=True)

In [ ]:
questions.count()

In [ ]:
questions.printSchema()

Объединим ее с ответами при условии, что эта запись ссылкается на вопрос если content_type_id и content_id - идентификатор вопроса.

In [ ]:
df = (
    riiid_df.
        where(f.col("content_type_id") == 0).
        join(questions, riiid_df.content_id == questions.question_id, 'left')
)

In [ ]:
df

Проверим, что вероястность правильного ответа не зависит от его номера.

In [ ]:
%%time
(
df
    .select('correct_answer', 'answered_correctly')
    .groupby('correct_answer')
    .mean('answered_correctly')
    .show()
)

## Упражнение 5

В файле "riiid/lectures.csv" хранится информация об лекциях. Объедините эту таблицу с основным набором данных при условии, что content_type_id == 1.